# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SabeenSaeed/machine_learning_projects/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Provisional lane: Refresh / Content Opportunity Scoring.** I want to help an SEO editor decide which content pages should receive a refresh review first. This lane fits the starter playground because it contains page-level observable signals such as impressions, clicks, CTR, position, content age, update recency, and an observed trend direction. I am choosing it provisionally because the starter pipeline already demonstrates a meaningful ranking problem, but I will revisit the lane after a fuller signal audit and leakage check. The goal is not to automate editing or claim to know Google's algorithm; it is to make a transparent, evidence-backed review queue.

In [1]:
# Load the real starter CSV from the repository, whether this notebook starts at
# the repo root or inside work/notebooks.
from pathlib import Path
import pandas as pd

repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "data/raw/content_refresh_anonymized.csv").exists()), None)
assert repo_root is not None, "Could not locate the starter CSV from the current working directory."
data_path = repo_root / "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_path)
df["is_declining_observed"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(f"Loaded {len(df):,} page rows and {df.shape[1]:,} columns from {data_path}")


Loaded 30,000 page rows and 45 columns from /home/ubuntu/machine_learning_projects/data/raw/content_refresh_anonymized.csv


## 2. The question: decision, action, cost of a wrong call

**Research question:** Among pages with comparable pre-decision search and content signals, which pages should an editor review first for a possible refresh, and can a simple ranked score prioritize them more usefully than a hand-written rule?

The **unit of analysis** is one content page at one snapshot in time. The output will be a ranked queue containing a priority score, the observable reason codes behind the score, and an uncertainty or review note. An SEO editor or content strategist would inspect the highest-ranked pages, check the underlying page context in their normal workflow, and decide whether to update, leave unchanged, or gather more evidence. A false positive sends an editor toward a page that does not need work and costs review time; a false negative leaves a genuinely declining opportunity lower in the queue and may delay a useful refresh. I will therefore measure ranking quality with Precision@K and inspect false-positive and false-negative trade-offs rather than optimize for a generic accuracy number.

In [2]:
# Define a transparent baseline for the proposed review queue.
# It is intentionally simple: stale pages that still have meaningful visibility
# are ranked by their observed exposure.
df["baseline_refresh_score"] = (
    df["days_since_last_update"].ge(180).astype(int)
    * df["impressions_90d"].fillna(0)
)
order = df["baseline_refresh_score"].sort_values(ascending=False).index
for k in (20, 50):
    precision = df.loc[order[:k], "is_declining_observed"].mean()
    print(f"Baseline Precision@{k}: {precision:.3f}")


Baseline Precision@20: 0.900
Baseline Precision@50: 0.680


## 3. Quick look at the data (2-3 real numbers)

The starter CSV contains enough page-level observations to make this worth investigating, while also showing why careful prioritization matters. The code below loads the real shipped dataset and reports the row count, the observed share of pages whose trend direction is `down`, and the share with at least 500 impressions in the 90-day window. These are descriptive facts about this snapshot, not causal claims. The `down` field is useful for early exploration, but a future project target should be defined from a later outcome window rather than treating a same-window derived label as proof of what a refresh caused.

In [3]:
# Three supporting numbers from the real starter dataset.
rows = len(df)
down_rate = df["is_declining_observed"].mean()
visible_rate = df["impressions_90d"].ge(500).mean()
print(f"Rows (page observations): {rows:,}")
print(f"Observed trend_direction == 'down': {down_rate:.3f} ({down_rate:.1%})")
print(f"Pages with impressions_90d >= 500: {visible_rate:.3f} ({visible_rate:.1%})")


Rows (page observations): 30,000
Observed trend_direction == 'down': 0.542 (54.2%)
Pages with impressions_90d >= 500: 0.558 (55.8%)


## 4. Careful words: what I can and can't claim

I can claim that this snapshot contains measurable differences among pages and that a ranking method can be compared with a transparent baseline on an explicitly defined metric. I can make **observed**, **directional**, and **decision-support** claims such as “the top K of this queue contained this share of pages with the observed down label in this evaluation.” I cannot claim that refreshes caused recovery, that a feature controls Google rankings, or that the model predicts Google's algorithm. Before treating a score as a recommendation, I will define a decision-time feature window, create a later outcome window, keep clients separated where appropriate, check leakage, report the base rate, and show uncertainty and limitations. A human editor remains responsible for the final action.

In [4]:
# Confirm that the proposed first-pass feature set uses observable signals,
# not product decision flags or identifiers as predictors.
feature_candidates = [
    "content_age_days", "days_since_last_update", "impressions_90d",
    "avg_position", "ctr", "word_count", "engagement_rate"
]
missing = [c for c in feature_candidates if c not in df.columns]
assert not missing, f"Missing expected observable columns: {missing}"
print("Observable feature candidates available:", ", ".join(feature_candidates))
print("Next validation requirement: define a decision-time window and a later observed outcome window before supervised modeling.")


Observable feature candidates available: content_age_days, days_since_last_update, impressions_90d, avg_position, ctr, word_count, engagement_rate
Next validation requirement: define a decision-time window and a later observed outcome window before supervised modeling.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.